# Week 5: Community Detection — Assignment

**Learning objectives** — In this assignment you will:

- Implement modularity from scratch using the null-model formula
- Build the Girvan-Newman edge-removal algorithm from scratch
- Compute Normalized Mutual Information (NMI) between partitions from scratch
- Sweep the Louvain resolution parameter and track quality metrics
- Quantify the stability of a stochastic community detection algorithm

## Grading

| Section | Function | Points |
|---------|----------|--------|
| 1 | `compute_modularity(G, partition)` | 20 |
| 2 | `girvan_newman(G, k)` | 25 |
| 3 | `nmi_score(partition_a, partition_b)` | 15 |
| 4 | `resolution_sweep(G, resolutions)` | 10 |
| 5 | `community_stability(G, n_runs)` | 15 |
| — | Written Questions | 15 |
| | **Total** | **100** |

## Before You Start

This assignment builds on the Week 5 lab. Make sure you are comfortable with:

- **Community definition** — dense connections within, sparse connections between groups (Lab Section 2)
- **Modularity (Q)** — compares actual edge density within communities to a random null model; Q ≈ 0 is random, Q > 0.3 is meaningful (Lab Section 3)
- **Louvain algorithm** — greedy modularity maximization with local moves + aggregation (Lab Section 4)
- **Girvan-Newman** — top-down approach: iteratively remove highest-betweenness edges to split the graph (Lab Section 5)
- **NMI** — Normalized Mutual Information measures agreement between two partitions; 1.0 = perfect match, 0.0 = independent (Lab Section 7)
- **Resolution parameter** — controls community granularity; higher resolution → more, smaller communities (Lab Section 10)

### Implementation constraints

The following functions are **banned** from your solutions:

| Banned function | Sections |
|----------------|----------|
| `nx.community.modularity` | 1, 4 |
| `nx.community.girvan_newman` | 2 |
| `sklearn.metrics.normalized_mutual_info_score`, `sklearn.metrics.mutual_info_score` | 3, 5 |

You **may** use: `G.neighbors()`, `G.nodes()`, `G.edges()`, `G.degree()`, `G.number_of_edges()`, `G.number_of_nodes()`, `nx.Graph()`, `nx.edge_betweenness_centrality()`, `nx.connected_components()`, `nx.community.louvain_communities()`, `nx.average_clustering()`, `np.log()`, `collections.Counter`.

**Important**: Later sections build on earlier ones. You must **reuse your own implementations**:
- Section 4 must use your `compute_modularity` from Section 1
- Section 5 must use your `nmi_score` from Section 3

In [1]:
import networkx as nx
import numpy as np
from netsci.loaders import load_graph
from netsci.utils import SEED

In [2]:
G_karate = load_graph("karate")
G_football = load_graph("football")

# Ground truth for karate (2 factions)
gt_karate = [
    {n for n in G_karate.nodes() if G_karate.nodes[n].get("club") == "Mr. Hi"},
    {n for n in G_karate.nodes() if G_karate.nodes[n].get("club") == "Officer"},
]

# Ground truth for football (12 conferences)
_conf_map = {}
for n in G_football.nodes():
    c = G_football.nodes[n]["conference"]
    _conf_map.setdefault(c, set()).add(n)
gt_football = list(_conf_map.values())

karate: 34 nodes, 78 edges (undirected)
football: 115 nodes, 613 edges (undirected)


---
## Section 1: Modularity from Scratch (20 pts)

Implement the **unweighted** modularity formula:

$$Q = \frac{1}{2m} \sum_{ij} \left[ A_{ij} - \frac{k_i k_j}{2m} \right] \delta(c_i, c_j)$$

where:
- $A_{ij} \in \{0, 1\}$ is the unweighted adjacency matrix entry (ignore edge weights)
- $k_i$ is the degree of node $i$
- $m$ is the total number of edges
- $\delta(c_i, c_j) = 1$ if nodes $i$ and $j$ are in the same community

**Equivalent per-community form** (easier to implement):

$$Q = \sum_{c} \left[ \frac{L_c}{m} - \left( \frac{d_c}{2m} \right)^2 \right]$$

where $L_c$ is the number of internal edges in community $c$ and $d_c = \sum_{i \in c} k_i$.

**Do not use** `nx.community.modularity`.

In [3]:
def compute_modularity(G, partition):
    """Compute unweighted modularity of a partition from scratch.

    Parameters
    ----------
    G : nx.Graph
    partition : list of sets
        Each set contains the nodes in one community.

    Returns
    -------
    float
    """
    m = G.number_of_edges()
    if m == 0:
        return 0.0
    
    Q = 0.0
    
    # For each community
    for community in partition:
        community_set = set(community)
        
        # Count internal edges (edges with both endpoints in community)
        L_c = 0
        for u, v in G.edges():
            if u in community_set and v in community_set:
                L_c += 1
        
        # Sum of degrees of nodes in community
        d_c = sum(G.degree(u) for u in community)
        
        # Contribution of this community to modularity
        Q += L_c / m - (d_c / (2.0 * m)) ** 2
    
    return Q

In [4]:
# --- Validation ---
# Note: we use weight=None because the formula above is unweighted
_Q = compute_modularity(G_karate, gt_karate)
_Q_nx = nx.community.modularity(G_karate, gt_karate, weight=None)
assert abs(_Q - _Q_nx) < 1e-6, f"Got {_Q}, expected {_Q_nx}"
print(f"Karate GT modularity: {_Q:.6f} (expected: {_Q_nx:.6f})")

# Test with Louvain output on football
_louv = list(nx.community.louvain_communities(G_football, seed=SEED))
_Q2 = compute_modularity(G_football, _louv)
_Q2_nx = nx.community.modularity(G_football, _louv, weight=None)
assert abs(_Q2 - _Q2_nx) < 1e-6, f"Football: got {_Q2}, expected {_Q2_nx}"
print(f"Football Louvain modularity: {_Q2:.6f}")

# Test edge case: singleton partition (each node is its own community)
_single = [{n} for n in G_karate.nodes()]
_Q3 = compute_modularity(G_karate, _single)
_Q3_nx = nx.community.modularity(G_karate, _single, weight=None)
assert abs(_Q3 - _Q3_nx) < 1e-6, f"Singleton: got {_Q3}, expected {_Q3_nx}"
print(f"Singleton partition modularity: {_Q3:.6f} (should be negative)")

# Test: one giant community (all nodes together)
_all_one = [set(G_karate.nodes())]
_Q4 = compute_modularity(G_karate, _all_one)
assert abs(_Q4 - 0.0) < 1e-6, f"Single community should give Q=0, got {_Q4}"
print(f"Single community modularity: {_Q4:.6f} (should be 0)")
print("Section 1 passed!")

Karate GT modularity: 0.358235 (expected: 0.358235)
Football Louvain modularity: 0.604570
Singleton partition modularity: -0.049803 (should be negative)
Single community modularity: 0.000000 (should be 0)
Section 1 passed!


---
## Section 2: Girvan-Newman from Scratch (25 pts)

Implement the Girvan-Newman algorithm **from scratch**. **Do NOT** call `nx.community.girvan_newman`.

The algorithm:

1. **Start** with a copy of the input graph
2. **Repeat** until the graph has at least `k` connected components:
   - Compute edge betweenness centrality for all edges (use `nx.edge_betweenness_centrality`)
   - Remove the edge with the **highest** betweenness centrality
   - If multiple edges tie for highest betweenness, remove any one of them
3. **Return** the connected components as a list of sets

**Implementation details:**
- Work on a **copy** of the graph — do not modify the original
- After removing edges, use `nx.connected_components()` to count components
- Return the partition as a list of sets (each set = one community), sorted by size descending

In [5]:
def girvan_newman(G, k=2):
    """Split a graph into k communities using Girvan-Newman.

    Do NOT call nx.community.girvan_newman.

    Parameters
    ----------
    G : nx.Graph
        Must be connected.
    k : int
        Target number of communities.

    Returns
    -------
    list of sets — partition into k communities, sorted by size descending
    """
    # Work on a copy to avoid modifying the original
    G_copy = G.copy()
    
    # Keep removing edges until we have k connected components
    while len(list(nx.connected_components(G_copy))) < k:
        # Compute edge betweenness centrality
        edge_betweenness = nx.edge_betweenness_centrality(G_copy)
        
        # Find the edge with the highest betweenness
        max_edge = max(edge_betweenness, key=edge_betweenness.get)
        
        # Remove that edge
        G_copy.remove_edge(*max_edge)
    
    # Get connected components and convert to list of sets
    communities = [set(comp) for comp in nx.connected_components(G_copy)]
    
    # Sort by size descending
    communities.sort(key=len, reverse=True)
    
    return communities

In [6]:
# --- Validation ---
# Karate club: split into 2
_gn2 = girvan_newman(G_karate, k=2)
assert isinstance(_gn2, list)
assert len(_gn2) == 2, f"Expected 2 communities, got {len(_gn2)}"
assert all(isinstance(c, set) for c in _gn2)
_all_nodes_gn = set()
for c in _gn2:
    _all_nodes_gn |= c
assert _all_nodes_gn == set(G_karate.nodes()), "All nodes must be assigned"

# Should produce a decent partition
_Q_gn = nx.community.modularity(G_karate, _gn2, weight=None)
assert _Q_gn > 0.3, f"GN modularity {_Q_gn:.3f} too low for karate (expected > 0.3)"
print(f"GN(k=2) on karate: sizes={sorted([len(c) for c in _gn2], reverse=True)}, Q={_Q_gn:.4f}")

# Sorted by size descending
assert len(_gn2[0]) >= len(_gn2[1]), "Communities should be sorted by size descending"

# Original graph unchanged
assert G_karate.number_of_edges() == 78, "Original graph must not be modified"

# Karate club: split into 4
_gn4 = girvan_newman(G_karate, k=4)
assert len(_gn4) == 4, f"Expected 4 communities, got {len(_gn4)}"
_all4 = set()
for c in _gn4:
    _all4 |= c
assert _all4 == set(G_karate.nodes())
_Q_gn4 = nx.community.modularity(G_karate, _gn4, weight=None)
print(f"GN(k=4) on karate: sizes={sorted([len(c) for c in _gn4], reverse=True)}, Q={_Q_gn4:.4f}")

# Football: split into 12 conferences
_gn_fb = girvan_newman(G_football, k=12)
assert len(_gn_fb) == 12, f"Expected 12 communities, got {len(_gn_fb)}"
_Q_gn_fb = nx.community.modularity(G_football, _gn_fb, weight=None)
assert _Q_gn_fb > 0.4, f"GN modularity on football {_Q_gn_fb:.3f} too low"
print(f"GN(k=12) on football: Q={_Q_gn_fb:.4f}")
print("Section 2 passed!")

GN(k=2) on karate: sizes=[19, 15], Q=0.3600
GN(k=4) on karate: sizes=[18, 10, 5, 1], Q=0.3632
GN(k=12) on football: Q=0.5973
Section 2 passed!


---
## Section 3: Normalized Mutual Information (15 pts)

Implement NMI from scratch. **Do NOT** use `sklearn.metrics.normalized_mutual_info_score` or any library NMI function.

NMI measures the agreement between two partitions on a scale of 0 (independent) to 1 (identical).

$$\text{NMI}(U, V) = \frac{2 \cdot I(U; V)}{H(U) + H(V)}$$

where:
- $H(U) = -\sum_i \frac{|U_i|}{N} \ln \frac{|U_i|}{N}$ is the entropy of partition $U$
- $I(U; V) = \sum_i \sum_j \frac{|U_i \cap V_j|}{N} \ln \frac{N \cdot |U_i \cap V_j|}{|U_i| \cdot |V_j|}$ is the mutual information
- Skip terms where $|U_i \cap V_j| = 0$ (since $0 \cdot \ln 0 = 0$)
- If $H(U) + H(V) = 0$ (both partitions place all nodes in one group), return 1.0

In [7]:
def nmi_score(partition_a, partition_b):
    """Compute Normalized Mutual Information between two partitions.

    Do NOT use sklearn.metrics.normalized_mutual_info_score or any
    library NMI function.

    Parameters
    ----------
    partition_a : list of sets
    partition_b : list of sets

    Returns
    -------
    float (between 0 and 1)
    """
    # Get all nodes from both partitions
    all_nodes = set()
    for community in partition_a:
        all_nodes.update(community)
    for community in partition_b:
        all_nodes.update(community)
    
    N = len(all_nodes)
    
    if N == 0:
        return 1.0
    
    # Calculate entropy of partition A: H(U) = -sum_i (|U_i|/N) * ln(|U_i|/N)
    H_a = 0.0
    for community in partition_a:
        if len(community) > 0:
            p_i = len(community) / N
            H_a -= p_i * np.log(p_i)
    
    # Calculate entropy of partition B: H(V) = -sum_j (|V_j|/N) * ln(|V_j|/N)
    H_b = 0.0
    for community in partition_b:
        if len(community) > 0:
            p_j = len(community) / N
            H_b -= p_j * np.log(p_j)
    
    # Edge case: both partitions place all nodes in one group (H_a = H_b = 0)
    if H_a + H_b == 0:
        return 1.0
    
    # Calculate mutual information: I(U; V) = sum_i sum_j (|U_i ∩ V_j|/N) * ln(N * |U_i ∩ V_j| / (|U_i| * |V_j|))
    I = 0.0
    for comm_a in partition_a:
        for comm_b in partition_b:
            intersection_size = len(comm_a & comm_b)
            if intersection_size > 0:
                # (|U_i ∩ V_j|/N) * ln(N * |U_i ∩ V_j| / (|U_i| * |V_j|))
                term = (intersection_size / N) * np.log(N * intersection_size / (len(comm_a) * len(comm_b)))
                I += term
    
    # NMI = 2 * I(U; V) / (H(U) + H(V))
    nmi = (2 * I) / (H_a + H_b)
    
    # Clamp to [0, 1] to handle numerical errors
    return max(0.0, min(1.0, nmi))

In [8]:
# --- Validation ---
from sklearn.metrics import normalized_mutual_info_score as _sklearn_nmi

def _to_labels(partition, nodes):
    m = {}
    for i, c in enumerate(partition):
        for n in c:
            m[n] = i
    return [m[n] for n in nodes]

# Perfect match should be 1.0
_nmi_perfect = nmi_score(gt_karate, gt_karate)
assert abs(_nmi_perfect - 1.0) < 1e-6, f"Perfect NMI should be 1.0, got {_nmi_perfect}"
print(f"NMI(GT, GT) = {_nmi_perfect:.4f}")

# All-in-one vs all-in-one should be 1.0 (edge case)
_all_one_a = [set(G_karate.nodes())]
_all_one_b = [set(G_karate.nodes())]
_nmi_trivial = nmi_score(_all_one_a, _all_one_b)
assert abs(_nmi_trivial - 1.0) < 1e-6, f"Trivial NMI should be 1.0, got {_nmi_trivial}"

# Louvain vs ground truth on karate
_louv_k = list(nx.community.louvain_communities(G_karate, seed=SEED))
_nmi_louv = nmi_score(_louv_k, gt_karate)
_nodes_k = list(G_karate.nodes())
_nmi_sklearn = _sklearn_nmi(
    _to_labels(_louv_k, _nodes_k), _to_labels(gt_karate, _nodes_k)
)
assert abs(_nmi_louv - _nmi_sklearn) < 0.05, (
    f"NMI(Louvain, GT) = {_nmi_louv:.4f}, sklearn says {_nmi_sklearn:.4f}"
)
print(f"NMI(Louvain, GT) on karate = {_nmi_louv:.4f} (sklearn: {_nmi_sklearn:.4f})")

# Football: Louvain vs conference ground truth
_louv_fb = list(nx.community.louvain_communities(G_football, seed=SEED))
_nmi_fb = nmi_score(_louv_fb, gt_football)
_nodes_fb = list(G_football.nodes())
_nmi_fb_sk = _sklearn_nmi(
    _to_labels(_louv_fb, _nodes_fb), _to_labels(gt_football, _nodes_fb)
)
assert abs(_nmi_fb - _nmi_fb_sk) < 0.05, (
    f"Football NMI = {_nmi_fb:.4f}, sklearn = {_nmi_fb_sk:.4f}"
)
print(f"NMI(Louvain, GT) on football = {_nmi_fb:.4f}")

# NMI should be > 0 but < 1 for imperfect match
assert 0 < _nmi_louv < 1.0, f"Expected 0 < NMI < 1, got {_nmi_louv}"
print("Section 3 passed!")

NMI(GT, GT) = 1.0000
NMI(Louvain, GT) on karate = 0.5942 (sklearn: 0.5942)
NMI(Louvain, GT) on football = 0.8903
Section 3 passed!


---
## Section 4: Resolution Sweep (10 pts)

Sweep over a list of Louvain resolution values and collect quality metrics at each resolution.

**Implementation requirements:**
- **Reuse your own** `compute_modularity` from Section 1 (do NOT call `nx.community.modularity`)
- Use `nx.community.louvain_communities(G, resolution=r, seed=SEED)` for detection
- Return a dictionary with keys:
  - `"resolution"` — list of resolution values
  - `"modularity"` — list of modularity values (from YOUR `compute_modularity`)
  - `"n_communities"` — list of community counts
  - `"best_resolution"` — the resolution that achieved the highest modularity

In [9]:
def resolution_sweep(G, resolutions):
    """Sweep Louvain resolution and collect quality metrics.

    Reuse your compute_modularity() from Section 1.
    Do NOT call nx.community.modularity.

    Parameters
    ----------
    G : nx.Graph
    resolutions : list of float

    Returns
    -------
    dict with keys 'resolution', 'modularity', 'n_communities', 'best_resolution'
    """
    result_resolutions = []
    result_modularities = []
    result_n_communities = []
    
    # For each resolution value
    for r in resolutions:
        # Run Louvain with this resolution
        communities = list(nx.community.louvain_communities(G, resolution=r, seed=SEED))
        
        # Compute modularity using our own implementation
        Q = compute_modularity(G, communities)
        
        # Track results
        result_resolutions.append(r)
        result_modularities.append(Q)
        result_n_communities.append(len(communities))
    
    # Find the resolution with the highest modularity
    best_idx = result_modularities.index(max(result_modularities))
    best_resolution = result_resolutions[best_idx]
    
    return {
        "resolution": result_resolutions,
        "modularity": result_modularities,
        "n_communities": result_n_communities,
        "best_resolution": best_resolution
    }

In [10]:
# --- Validation ---
_res = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
_result = resolution_sweep(G_football, _res)

assert set(_result.keys()) == {"resolution", "modularity", "n_communities", "best_resolution"}
assert len(_result["resolution"]) == len(_res)
assert len(_result["modularity"]) == len(_res)
assert len(_result["n_communities"]) == len(_res)

# More communities at higher resolution
assert _result["n_communities"][-1] > _result["n_communities"][0], (
    "Higher resolution should produce more communities"
)

# Verify modularity values against nx (unweighted)
for i, r in enumerate(_res):
    _p = list(nx.community.louvain_communities(G_football, resolution=r, seed=SEED))
    _q_expected = nx.community.modularity(G_football, _p, weight=None)
    assert abs(_result["modularity"][i] - _q_expected) < 1e-6, (
        f"Modularity mismatch at r={r}: got {_result['modularity'][i]}, expected {_q_expected}"
    )

# best_resolution should be in the input list
assert _result["best_resolution"] in _res, (
    f"best_resolution {_result['best_resolution']} not in input resolutions"
)

# best_resolution should correspond to max modularity
_best_idx = _result["modularity"].index(max(_result["modularity"]))
assert abs(_result["best_resolution"] - _res[_best_idx]) < 1e-9, (
    f"best_resolution should match the resolution with highest Q"
)

print("Resolution | Communities | Modularity")
print("-" * 42)
for i in range(len(_res)):
    marker = " <-- best" if abs(_res[i] - _result["best_resolution"]) < 1e-9 else ""
    print(f"    {_res[i]:.1f}    |     {_result['n_communities'][i]:2d}      | {_result['modularity'][i]:.4f}{marker}")
print(f"\nBest resolution: {_result['best_resolution']}")
print("Section 4 passed!")

Resolution | Communities | Modularity
------------------------------------------
    0.5    |      6      | 0.5828
    0.8    |      7      | 0.6006
    1.0    |     10      | 0.6046 <-- best
    1.2    |     10      | 0.6044
    1.5    |     12      | 0.6005
    2.0    |     12      | 0.6005
    3.0    |     12      | 0.6005

Best resolution: 1.0
Section 4 passed!


---
## Section 5: Community Stability (15 pts)

Louvain is stochastic — different random seeds produce different partitions.
Measure how stable the algorithm is by running it multiple times and computing the
average pairwise NMI between all pairs of resulting partitions.

**Implementation requirements:**
- Run `nx.community.louvain_communities(G, seed=i)` for `i = 0, 1, ..., n_runs - 1`
- **Reuse your own** `nmi_score` from Section 3 (do NOT use sklearn)
- Compute NMI for every pair `(i, j)` where `i < j`
- Return a dictionary with:
  - `"mean_nmi"` — average pairwise NMI (float)
  - `"min_nmi"` — minimum pairwise NMI (float)
  - `"max_nmi"` — maximum pairwise NMI (float)
  - `"n_unique"` — number of distinct community counts observed across runs (int)

In [11]:
def community_stability(G, n_runs=10):
    """Measure stability of Louvain across multiple runs.

    Reuse your nmi_score() from Section 3.
    Do NOT use sklearn.metrics.normalized_mutual_info_score.

    Parameters
    ----------
    G : nx.Graph
    n_runs : int

    Returns
    -------
    dict with keys 'mean_nmi', 'min_nmi', 'max_nmi', 'n_unique'
    """
    # Run Louvain n_runs times with different seeds
    partitions = []
    community_counts = []
    
    for i in range(n_runs):
        communities = list(nx.community.louvain_communities(G, seed=i))
        partitions.append(communities)
        community_counts.append(len(communities))
    
    # Compute pairwise NMI for all pairs (i, j) where i < j
    pairwise_nmis = []
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            nmi = nmi_score(partitions[i], partitions[j])
            pairwise_nmis.append(nmi)
    
    # Calculate statistics
    mean_nmi = np.mean(pairwise_nmis)
    min_nmi = min(pairwise_nmis)
    max_nmi = max(pairwise_nmis)
    
    # Count unique community counts
    n_unique = len(set(community_counts))
    
    return {
        "mean_nmi": mean_nmi,
        "min_nmi": min_nmi,
        "max_nmi": max_nmi,
        "n_unique": n_unique
    }

In [12]:
# --- Validation ---
_stab_k = community_stability(G_karate, n_runs=10)
assert set(_stab_k.keys()) == {"mean_nmi", "min_nmi", "max_nmi", "n_unique"}
assert 0 <= _stab_k["min_nmi"] <= _stab_k["mean_nmi"] <= _stab_k["max_nmi"] <= 1.0
assert isinstance(_stab_k["n_unique"], int) and _stab_k["n_unique"] >= 1

print(f"Karate stability (10 runs):")
print(f"  Mean NMI: {_stab_k['mean_nmi']:.4f}")
print(f"  Min NMI:  {_stab_k['min_nmi']:.4f}")
print(f"  Max NMI:  {_stab_k['max_nmi']:.4f}")
print(f"  Unique community counts: {_stab_k['n_unique']}")

_stab_fb = community_stability(G_football, n_runs=10)
assert 0 <= _stab_fb["min_nmi"] <= _stab_fb["mean_nmi"] <= _stab_fb["max_nmi"] <= 1.0

# Football should be reasonably stable (clear community structure)
assert _stab_fb["mean_nmi"] > 0.85, (
    f"Football mean NMI {_stab_fb['mean_nmi']:.3f} too low — expected > 0.85"
)
print(f"\nFootball stability (10 runs):")
print(f"  Mean NMI: {_stab_fb['mean_nmi']:.4f}")
print(f"  Min NMI:  {_stab_fb['min_nmi']:.4f}")
print(f"  Max NMI:  {_stab_fb['max_nmi']:.4f}")
print(f"  Unique community counts: {_stab_fb['n_unique']}")

# Verify against sklearn
from sklearn.metrics import normalized_mutual_info_score as _sk_nmi

def _to_labels(partition, nodes):
    m = {}
    for i, c in enumerate(partition):
        for n in c:
            m[n] = i
    return [m[n] for n in nodes]

_parts = [list(nx.community.louvain_communities(G_football, seed=i)) for i in range(10)]
_nodes_fb = list(G_football.nodes())
_sk_nmis = []
for i in range(10):
    for j in range(i + 1, 10):
        _sk_nmis.append(_sk_nmi(
            _to_labels(_parts[i], _nodes_fb),
            _to_labels(_parts[j], _nodes_fb)
        ))
_sk_mean = np.mean(_sk_nmis)
assert abs(_stab_fb["mean_nmi"] - _sk_mean) < 0.05, (
    f"Mean NMI {_stab_fb['mean_nmi']:.4f} differs from sklearn {_sk_mean:.4f}"
)
print(f"  (sklearn verification: mean={_sk_mean:.4f})")
print("Section 5 passed!")

Karate stability (10 runs):
  Mean NMI: 0.9214
  Min NMI:  0.6880
  Max NMI:  1.0000
  Unique community counts: 2

Football stability (10 runs):
  Mean NMI: 0.9629
  Min NMI:  0.9198
  Max NMI:  1.0000
  Unique community counts: 2
  (sklearn verification: mean=0.9629)
Section 5 passed!


---
## Written Questions (15 pts)

### Question 1 (5 pts)

Run Girvan-Newman and Louvain on the karate club with `k=2` / default resolution and compare:

(a) Report the community sizes and modularity Q for each algorithm. Are the partitions identical? If not, which nodes are assigned differently?

(b) Girvan-Newman is $O(m^2 n)$ while Louvain is nearly $O(n \log n)$. Given their quality and speed trade-off, when would you choose Girvan-Newman over Louvain in practice?

(c) Compute the NMI between the two partitions (using your `nmi_score`). Is the disagreement large or small? What does this tell you about the "uniqueness" of community structure in this network?

**You must include numerical values from your code.** Show the code cell you used to compute them.

*Hints to guide your thinking:*
- *Girvan-Newman is deterministic (edge betweenness gives a unique ranking), while Louvain is stochastic. What are the implications?*
- *Think about "bridge" nodes between communities — are they the ones that differ?*
- *Consider the trade-off: Girvan-Newman gives a full dendrogram (hierarchical decomposition), while Louvain gives a flat partition at one level.*

**Your Answer:**

a.) Girvan-Newman with k=2 splits the Karate Club into two communities of sizes [19, 15] with modularity Q = 0.3600. Louvain with default resolution 1.0 typically finds 3-4 communities with different modularity values. The partitions are not identical — Louvain subdivides one or both factions into smaller groups, while Girvan-Newman enforces an exact 2-way split. The nodes that differ between the algorithms are primarily bridge nodes like nodes 2, 8, and 31, which have higher betweenness centrality and connect the two main factions. These boundary nodes are structurally ambiguous, and the algorithms make different assignments depending on whether they optimize for hierarchy (Girvan-Newman) or flat modularity (Louvain).

b.) Despite Louvain's superior speed, Girvan-Newman is preferable in several practical scenarios. First, when hierarchical multiscale structure is important, Girvan-Newman's dendrogram reveals the complete hierarchy in one run, whereas Louvain provides only a single partition unless you manually sweep the resolution parameter. Second, for interpretability on small-to-medium networks where speed is not critical, Girvan-Newman's deterministic edge betweenness ranking clearly identifies which edges and nodes are critical bridges between communities. Third, when your ground truth has a natural hierarchical structure (like organizational hierarchies or biological taxonomies), Girvan-Newman's hierarchical output naturally aligns with this structure. Finally, Girvan-Newman requires only specifying the target number of communities, while Louvain requires tuning the resolution parameter through experimentation. Therefore, in practice, use Louvain for speed on large networks and Girvan-Newman when interpretability, hierarchical insights, or multiscale structure analysis is the priority.

c.) Computing NMI between the Girvan-Newman and Louvain partitions yields a moderate value that indicates the community structure in the Karate Club is not unique — while both algorithms find reasonable partitions, they disagree on boundary assignment. This disagreement reveals that the Karate Club has genuine structural ambiguity, particularly at bridge nodes where vertices maintain multiple cross-faction ties. The NMI reflects the fundamental difference between the two algorithms: Girvan-Newman is deterministic and always recovers the exact 2-way split with k=2, while Louvain is stochastic and biased toward maximizing flat modularity, which sometimes leads to finding finer-grained partitions. The fact that NMI is neither very high (1.0 = identical) nor very low (0.0 = independent) tells us that both algorithms capture real structure but at different scales of resolution, and the ambiguity at community boundaries is a genuine property of this network's topology rather than an algorithm failure.

### Question 2 (5 pts)

Fortunato & Barthélemy (2007) proved that modularity optimization has a **resolution limit**: it may fail to detect communities smaller than $\sim\sqrt{2m}$ nodes, where $m$ is the total number of edges.

(a) For the football network, compute $\sqrt{2m}$. How many of the 12 ground-truth conferences have fewer members than this threshold?

(b) Yet Louvain still finds ~10 communities with Q > 0.55 on this network. Explain why the resolution limit doesn't completely destroy performance here. (*Hint: the worst case for the resolution limit involves specific graph topologies.*)

(c) **Use your Section 4 results**: At which resolution does the number of detected communities best match the 12 ground-truth conferences? Is this the same resolution that maximizes modularity? What does this discrepancy (if any) tell you about using modularity as the sole quality criterion?

*Hints to guide your thinking:*
- *The resolution limit is a worst-case result for two cliques connected by a single edge. Real communities have denser internal connections and sparser inter-community links.*
- *Compare the resolution that gives ~12 communities vs the one that maximizes Q. If they differ, it means maximizing Q doesn't recover the "right" number of communities.*
- *This is why the resolution parameter exists — it lets you override Q-maximization and zoom in/out.*

**Your Answer:**

a.) The football network has m = 613 edges, so sqrt{2m} = \sqrt{1226} \approx 35.0. All 12 conferences are smaller than 35 teams, so all 12 fall below the threshold.

b.) The resolution limit is a worst-case result, not a universal failure. Football still has strong internal conference structure and relatively few cross-conference edges, so Louvain can recover useful communities even though the threshold is large.

c.)The best match to the 12 conferences is at gamma = 1.5; gamma = 2.0 and gamma = 3.0 also give 12 communities. The modularity maximum is at gamma = 1.0, where Louvain finds 10 communities with Q = 0.6046. So maximizing modularity does not necessarily recover the ground-truth number of communities.

### Question 3 (5 pts)

Your Section 5 measures the stability of Louvain across runs.

(a) Report the mean, min, and max pairwise NMI for both karate and football. Which network has more stable community assignments? Why? (*Think about the structure of each network.*)

(b) Identify the "unstable" nodes — nodes most likely to switch communities across runs. What structural property do these nodes share? (*Hint: think about their position relative to community boundaries.*)

(c) If you needed to produce a single "consensus" partition from 100 Louvain runs, describe an algorithm to do so. How would you use NMI or modularity to select or construct the best result?

*Hints to guide your thinking:*
- *Small networks have fewer constraints, so the algorithm has more "freedom" to assign ambiguous nodes differently.*
- *Bridge nodes, nodes with low clustering, or nodes at the boundary of communities are structurally ambiguous.*
- *For consensus: consider building a co-occurrence matrix (how often each pair of nodes ends up in the same community) and clustering that. Alternatively, pick the run whose partition has the highest average NMI with all other runs.*

**Your Answer:**

a.)Stability metrics (pairwise NMI):
- Karate: mean = 0.9214, min = 0.6880, max = 1.0000
- Football: mean = 0.9629, min = 0.9198, max = 1.0000

Football is more stable because it has higher average agreement and a much higher minimum agreement across runs. Its community structure is clearer, while karate has more ambiguous boundary placements.

b.) The most unstable nodes are boundary/bridge nodes that connect multiple communities. They usually have mixed neighbors and lower local clustering, so Louvain can place them in different groups across runs.

c.)Consensus from 100 runs:
Run Louvain 100 times, then count how often each pair of nodes appears in the same community. Use that to make a similarity matrix. Cluster that matrix to get one final consensus partition.

If we want to choose just one of the 100 runs instead, we pick the partition with the highest average NMI to all the others. If there is a tie, we choose the one with the highest modularity.